# IMDB Sentiment Classification — Distilled Pipeline (target: 0.94+)

This notebook upgrades the 0.92 baseline with five orthogonal techniques. Each is gated by a flag so you can ablate.

1. **TAPT** — Task-Adaptive Pretraining. Continued MLM on `train + test + unsup` text before fine-tuning. Adapts BERT-Tiny to IMDB vocabulary.
2. **Knowledge Distillation** — Train RoBERTa teacher on `train.csv`, generate soft probabilities for all available text, distill into the BERT-Tiny student.
3. **R-Drop** — Two forward passes with different dropout per step + symmetric KL between them. Strong regularizer.
4. **FGM** — Fast Gradient Method adversarial training on the embedding layer.
5. **EMA** — Exponential moving average of student weights. Use the EMA copy at inference.

The student stays at **9.59M parameters** (BERT-Tiny `L-2_H-256_A-4`). The teacher is a separate model used only to *generate training labels* — it is never deployed and does not count against the 10M budget.

### Required data layout
```
imdb-review-classification/
    train.csv                # 45k labeled
    test.csv                 # 5k unlabeled (predict these)
aclImdb/
    train/
        unsup/               # 50k unlabeled .txt files
```

If you don't have `aclImdb/train/unsup`, the pipeline still runs — it just uses `train + test` text for TAPT and skips the unsup half of distillation.

## 1. Setup

In [ ]:
from pathlib import Path
import random
import re
import html
import copy
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split, StratifiedKFold
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForMaskedLM,
    AutoModelForSequenceClassification,
    DataCollatorForLanguageModeling,
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup,
)

SEED = 42

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Master switches — turn things off here to ablate
USE_TAPT          = True
USE_DISTILLATION  = True
USE_RDROP         = True
USE_FGM           = True
USE_EMA           = True
N_ENSEMBLE_SEEDS  = 10
USE_OOF_TEACHER = True
TEACHER_N_FOLDS = 5

# Where to cache intermediate artifacts
ART_DIR = Path("artifacts")
ART_DIR.mkdir(exist_ok=True)

## 2. Load and clean data

Same cleaner as before — strip HTML tags, decode entities, collapse repeated punctuation.

In [ ]:
DATA_DIR = Path("imdb-review-classification")
UNSUP_DIR = Path("aclImdb/train/unsup")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

def clean_review(text):
    text = str(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = html.unescape(text)
    text = re.sub(r'([!?.,])\1{2,}', r'\1\1', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

train_df["review"] = train_df["review"].apply(clean_review)
test_df["review"]  = test_df["review"].apply(clean_review)

train_df.columns = train_df.columns.str.strip().str.lower()
test_df.columns  = test_df.columns.str.strip().str.lower()
train_df["label"] = train_df["label"].astype(str).str.strip().str.lower()

print("Train shape:", train_df.shape)
print("Test shape: ", test_df.shape)
print(train_df["label"].value_counts())

# Optional unlabeled corpus
if UNSUP_DIR.exists():
    unsup_texts = [f.read_text(encoding="utf-8") for f in sorted(UNSUP_DIR.glob("*.txt"))]
    unsup_df = pd.DataFrame({"review": unsup_texts})
    unsup_df["review"] = unsup_df["review"].apply(clean_review)
    HAS_UNSUP = True
    print(f"Loaded {len(unsup_df)} unlabeled reviews from aclImdb/train/unsup")
else:
    unsup_df = pd.DataFrame({"review": []})
    HAS_UNSUP = False
    print("No aclImdb/train/unsup found — skipping unsup half of distillation")

## 3. Student tokenizer & dataset

The student dataset now also carries optional `teacher_probs` (positive-class probability from the teacher) when distillation is enabled. Hard labels carry `-1` for unsup rows so the loss function can skip them.

In [ ]:
STUDENT_NAME = "google/bert_uncased_L-2_H-256_A-4"   # BERT-Tiny, 9.59M params
MAX_LEN = 512

student_tokenizer = AutoTokenizer.from_pretrained(STUDENT_NAME, use_fast=True)
print("Student vocab size:", len(student_tokenizer))

label_to_num = {"negative": 0, "positive": 1}
num_to_label = {0: "negative", 1: "positive"}


class IMDBDataset(Dataset):
    """
    Holds reviews + (optionally) hard labels + (optionally) teacher_pos_prob.
    
    label_int: 0 / 1 for labeled rows, -1 for unsup rows (skipped in CE loss).
    teacher_pos_prob: float in [0, 1] for distillation, or NaN if unavailable.
    """
    def __init__(self, df, tokenizer, max_len=MAX_LEN, mode="head"):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        review = str(row["review"])

        if self.mode == "tail":
            full = self.tokenizer(review, add_special_tokens=False)["input_ids"]
            keep = full[-(self.max_len - 2):]
            ids = [self.tokenizer.cls_token_id] + keep + [self.tokenizer.sep_token_id]
            attn = [1] * len(ids)
        elif self.mode == "middle":
            full = self.tokenizer(review, add_special_tokens=False)["input_ids"]
            inner = self.max_len - 2
            if len(full) <= inner:
                keep = full
            else:
                start = (len(full) - inner) // 2
                keep = full[start:start + inner]
            ids = [self.tokenizer.cls_token_id] + keep + [self.tokenizer.sep_token_id]
            attn = [1] * len(ids)
        else:  # "head"
            enc = self.tokenizer(review, max_length=self.max_len,
                                 truncation=True, padding=False,
                                 return_attention_mask=True)
            ids, attn = enc["input_ids"], enc["attention_mask"]

        item = {
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
        }

        # Hard label (or -1 sentinel for unsup)
        if "label_int" in row and row["label_int"] is not None:
            item["labels"] = torch.tensor(int(row["label_int"]), dtype=torch.long)
        else:
            item["labels"] = torch.tensor(-1, dtype=torch.long)

        # Teacher soft label (positive-class probability), NaN if unavailable
        if "teacher_pos_prob" in row and not pd.isna(row["teacher_pos_prob"]):
            item["teacher_pos_prob"] = torch.tensor(float(row["teacher_pos_prob"]),
                                                    dtype=torch.float)
        else:
            item["teacher_pos_prob"] = torch.tensor(float("nan"), dtype=torch.float)

        return item


def make_collate(tokenizer):
    pad_id = tokenizer.pad_token_id
    def collate_fn(batch):
        max_l = max(item["input_ids"].size(0) for item in batch)
        input_ids, attn, labels, teacher_pp = [], [], [], []
        for item in batch:
            L = item["input_ids"].size(0)
            pad = max_l - L
            input_ids.append(F.pad(item["input_ids"], (0, pad), value=pad_id))
            attn.append(F.pad(item["attention_mask"], (0, pad), value=0))
            labels.append(item["labels"])
            teacher_pp.append(item["teacher_pos_prob"])
        return {
            "input_ids": torch.stack(input_ids),
            "attention_mask": torch.stack(attn),
            "labels": torch.stack(labels),
            "teacher_pos_prob": torch.stack(teacher_pp),
        }
    return collate_fn

student_collate = make_collate(student_tokenizer)

## 4. Student model: BERT-Tiny + mean pooling

Same architecture as before. Two extras:
- `load_encoder_state(path)` for loading TAPT-pretrained weights.
- Slightly higher dropout (0.3) since R-Drop benefits from larger dropout values.

In [ ]:
class MeanPoolBertClassifier(nn.Module):
    """
    BERT-Tiny classifier with CLS + mean + max pooling.

    This usually works better than mean pooling alone and barely changes
    the parameter count.
    """
    def __init__(self, model_name, num_labels=2, dropout=0.3):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        hidden = self.backbone.config.hidden_size
        self.dropout = nn.Dropout(dropout)

        # CLS + mean + max = 3 * hidden
        self.classifier = nn.Linear(hidden * 3, num_labels)

    def forward(self, input_ids, attention_mask, **kwargs):
        out = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        last = out.last_hidden_state

        mask = attention_mask.unsqueeze(-1).float()

        # CLS pooling
        cls_pool = last[:, 0]

        # Mean pooling
        mean_pool = (last * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1e-9)

        # Max pooling, ignoring padding
        masked_last = last.masked_fill(mask == 0, -1e4)
        max_pool = masked_last.max(dim=1).values

        pooled = torch.cat([cls_pool, mean_pool, max_pool], dim=-1)
        pooled = self.dropout(pooled)

        return {"logits": self.classifier(pooled)}

    def load_encoder_state(self, path):
        """Load TAPT-pretrained backbone weights."""
        sd = torch.load(path, map_location="cpu")
        missing, unexpected = self.backbone.load_state_dict(sd, strict=False)
        print(f"  loaded TAPT weights | missing={len(missing)} unexpected={len(unexpected)}")


# Sanity check
_tmp = MeanPoolBertClassifier(STUDENT_NAME)
n_total = sum(p.numel() for p in _tmp.parameters())
n_train = sum(p.numel() for p in _tmp.parameters() if p.requires_grad)
print(f"Student total params:     {n_total:,}")
print(f"Student trainable params: {n_train:,}")
assert n_total < 10_000_000, "Over 10M parameter budget!"
del _tmp

## 5. TAPT — Continued MLM Pretraining

We continue pretraining the BERT-Tiny encoder with masked language modeling on **all available reviews** (`train + test + unsup`). Test reviews are fair game here because we only use the *text*, never the labels.

Why this helps: BERT-Tiny was pretrained on Wikipedia + BooksCorpus. IMDB has its own vocabulary — actor names, movie titles, "spoiler," "plot twist," "Tarantino," etc. Continued pretraining adapts the embeddings and attention patterns to that distribution. Typical gain: +0.5-1.0% on IMDB at small scales.

We save only the `bert.*` weights (encoder + embeddings) so they slot directly into the classifier.

In [ ]:
TAPT_OUT = ART_DIR / "tapt_encoder.pt"
TAPT_EPOCHS = 3
TAPT_LR = 5e-5
TAPT_BATCH = 64
TAPT_MASK_PROB = 0.15

def run_tapt():
    print("=== Running TAPT ===")
    
    # Build the unlabeled corpus: train text + test text + unsup text
    corpus_texts = (
        list(train_df["review"]) +
        list(test_df["review"]) +
        list(unsup_df["review"])
    )
    print(f"TAPT corpus size: {len(corpus_texts):,} reviews")
    
    # Tokenize once, store as fixed-length blocks for MLM
    class MLMDataset(Dataset):
        def __init__(self, texts, tokenizer, max_len=MAX_LEN):
            self.texts = texts
            self.tokenizer = tokenizer
            self.max_len = max_len
        def __len__(self):
            return len(self.texts)
        def __getitem__(self, idx):
            enc = self.tokenizer(self.texts[idx], max_length=self.max_len,
                                 truncation=True, padding=False,
                                 return_special_tokens_mask=True)
            return {
                "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
                "special_tokens_mask": torch.tensor(enc["special_tokens_mask"], dtype=torch.long),
            }
    
    mlm_collator = DataCollatorForLanguageModeling(
        tokenizer=student_tokenizer, mlm=True, mlm_probability=TAPT_MASK_PROB,
    )
    
    ds = MLMDataset(corpus_texts, student_tokenizer)
    loader = DataLoader(ds, batch_size=TAPT_BATCH, shuffle=True,
                    collate_fn=mlm_collator, num_workers=0)
    
    mlm_model = AutoModelForMaskedLM.from_pretrained(STUDENT_NAME).to(device)
    optimizer = torch.optim.AdamW(mlm_model.parameters(), lr=TAPT_LR, weight_decay=0.01)
    
    total_steps = len(loader) * TAPT_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps,
    )
    
    mlm_model.train()
    step = 0
    for epoch in range(TAPT_EPOCHS):
        running, n = 0.0, 0
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            out = mlm_model(**batch)
            out.loss.backward()
            torch.nn.utils.clip_grad_norm_(mlm_model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            running += out.loss.item() * batch["input_ids"].size(0)
            n += batch["input_ids"].size(0)
            step += 1
            if step % 200 == 0:
                print(f"  TAPT epoch {epoch+1} step {step} | loss {running/n:.4f}")
        print(f"TAPT epoch {epoch+1} done | avg loss {running/n:.4f}")
    
    # Save just the encoder weights (bert.* sub-state)
    backbone_state = {k.replace("bert.", "", 1): v.cpu()
                      for k, v in mlm_model.state_dict().items()
                      if k.startswith("bert.")}
    torch.save(backbone_state, TAPT_OUT)
    print(f"Saved TAPT encoder to {TAPT_OUT}")
    
    del mlm_model
    torch.cuda.empty_cache()


if USE_TAPT and not TAPT_OUT.exists():
    run_tapt()
elif USE_TAPT:
    print(f"Re-using existing TAPT artifact at {TAPT_OUT}")
else:
    print("Skipping TAPT (USE_TAPT=False)")

## 6. Teacher model — fine-tune RoBERTa on IMDB train

The teacher is a strong external model used **only to generate training signal** for the student. It is not deployed, not submitted, and does not count against the 10M budget.

Three options, in order of strength:
- **A.** `roberta-large` fine-tuned on `train.csv` (~96.5% on IMDB) — strongest teacher, but needs 16+ GB GPU.
- **B.** `roberta-base` fine-tuned on `train.csv` (~95.5% on IMDB) — good balance, default here.
- **C.** Off-the-shelf `textattack/roberta-base-imdb` — no training, ~95% on IMDB.

Set `TEACHER_MODE` below. The default (B) takes ~30-60 min on a T4.

In [ ]:
TEACHER_MODE = "D"   # "A" roberta-large, "B" roberta-base, "C" off-shelf, "D" deberta-v3-large

if TEACHER_MODE == "A":
    TEACHER_NAME = "roberta-large"
    TEACHER_CKPT = ART_DIR / "teacher_roberta_large.pt"
    TEACHER_EPOCHS = 3
    TEACHER_LR = 1e-5
    TEACHER_BATCH = 8
    TEACHER_MAX_LEN = 512
    TRAIN_TEACHER = True

elif TEACHER_MODE == "B":
    TEACHER_NAME = "roberta-base"
    TEACHER_CKPT = ART_DIR / "teacher_roberta_base.pt"
    TEACHER_EPOCHS = 3
    TEACHER_LR = 2e-5
    TEACHER_BATCH = 16
    TEACHER_MAX_LEN = 512
    TRAIN_TEACHER = True

elif TEACHER_MODE == "C":
    TEACHER_NAME = "textattack/roberta-base-imdb"
    TEACHER_CKPT = None
    TEACHER_MAX_LEN = 512
    TEACHER_BATCH = 32
    TRAIN_TEACHER = False

elif TEACHER_MODE == "D":
    TEACHER_NAME = "microsoft/deberta-v3-large"
    TEACHER_CKPT = ART_DIR / "teacher_deberta_v3_large.pt"
    TEACHER_EPOCHS = 3
    TEACHER_LR = 8e-6
    TEACHER_BATCH = 2
    TEACHER_MAX_LEN = 512
    TRAIN_TEACHER = True

else:
    raise ValueError(TEACHER_MODE)

TEACHER_SCORE_BATCH = max(1, TEACHER_BATCH * 2)

print("Teacher:", TEACHER_NAME, "| train teacher?", TRAIN_TEACHER)

In [ ]:
# === Teacher fine-tuning (Modes A & B only) ===

def train_teacher():
    print(f"=== Fine-tuning teacher {TEACHER_NAME} on train.csv ===")
    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME, use_fast=True)
    
    class TeacherDS(Dataset):
        def __init__(self, df, tokenizer, max_len, has_labels=True):
            self.df = df.reset_index(drop=True)
            self.tokenizer = tokenizer
            self.max_len = max_len
            self.has_labels = has_labels
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            enc = self.tokenizer(str(row["review"]), max_length=self.max_len,
                                 truncation=True, padding=False)
            item = {
                "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
            }
            if self.has_labels:
                item["labels"] = torch.tensor(label_to_num[row["label"]], dtype=torch.long)
            return item
    
    def teacher_collate(batch):
        max_l = max(b["input_ids"].size(0) for b in batch)
        pad_id = teacher_tokenizer.pad_token_id
        ids, attn, labels = [], [], []
        for b in batch:
            L = b["input_ids"].size(0); pad = max_l - L
            ids.append(F.pad(b["input_ids"], (0, pad), value=pad_id))
            attn.append(F.pad(b["attention_mask"], (0, pad), value=0))
            if "labels" in b:
                labels.append(b["labels"])
        out = {"input_ids": torch.stack(ids), "attention_mask": torch.stack(attn)}
        if labels:
            out["labels"] = torch.stack(labels)
        return out
    
    # 95/5 split for early stopping
    tr, va = train_test_split(train_df, test_size=0.05, random_state=SEED,
                              stratify=train_df["label"])
    
    train_ds = TeacherDS(tr, teacher_tokenizer, TEACHER_MAX_LEN, has_labels=True)
    val_ds   = TeacherDS(va, teacher_tokenizer, TEACHER_MAX_LEN, has_labels=True)
    
    train_loader = DataLoader(train_ds, batch_size=TEACHER_BATCH, shuffle=True,
                              collate_fn=teacher_collate, num_workers=0)
    val_loader = DataLoader(val_ds, batch_size=TEACHER_BATCH * 2, shuffle=False,
                            collate_fn=teacher_collate, num_workers=0)
    
    model = AutoModelForSequenceClassification.from_pretrained(TEACHER_NAME, num_labels=2).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=TEACHER_LR, weight_decay=0.01)
    total_steps = len(train_loader) * TEACHER_EPOCHS
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(0.06 * total_steps),
        num_training_steps=total_steps,
    )
    
    # Mixed precision so RoBERTa-large is feasible on smaller GPUs
    scaler = torch.cuda.amp.GradScaler()
    use_amp = TEACHER_MODE in ("A", "B")
    
    best_val = 0.0
    for epoch in range(TEACHER_EPOCHS):
        model.train()
        running, correct, total = 0.0, 0, 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(**batch)
                loss = out.loss
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            bs = batch["labels"].size(0)
            running += loss.item() * bs
            correct += (out.logits.argmax(-1) == batch["labels"]).sum().item()
            total += bs
        train_acc = correct / total
        
        # Validate
        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.cuda.amp.autocast(enabled=use_amp):
                    out = model(**batch)
                v_correct += (out.logits.argmax(-1) == batch["labels"]).sum().item()
                v_total += batch["labels"].size(0)
        val_acc = v_correct / v_total
        print(f"Teacher epoch {epoch+1}: train_acc={train_acc:.4f} val_acc={val_acc:.4f}")
        
        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), TEACHER_CKPT)
            print(f"  saved teacher checkpoint (val_acc={val_acc:.4f})")
    
    del model
    torch.cuda.empty_cache()
    return teacher_tokenizer


teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME, use_fast=True)

if USE_DISTILLATION and TRAIN_TEACHER:
    if TEACHER_CKPT.exists():
        print(f"Re-using existing teacher at {TEACHER_CKPT}")
    else:
        teacher_tokenizer = train_teacher()
elif USE_DISTILLATION:
    print(f"Using off-the-shelf teacher: {TEACHER_NAME}")
else:
    print("Skipping teacher (USE_DISTILLATION=False)")

## 7. Generate teacher soft labels

We score every available review (`train + unsup`) with the teacher and store the **positive-class probability**. These become the soft targets for the student. We do not need teacher predictions on `test.csv` — that's what the student is for at the end.

We use 2-crop TTA (head + tail) on the teacher too, for cleaner soft labels.

In [ ]:
TEACHER_PROBS_PATH = ART_DIR / f"teacher_probs_oof_{TEACHER_MODE}_{TEACHER_N_FOLDS}.npz"

def teacher_score_corpus(df, tokenizer, model, max_len=TEACHER_MAX_LEN, batch_size=32):
    """Return positive-class probabilities, head/tail averaged."""
    
    class ScoreDS(Dataset):
        def __init__(self, df, tok, max_len, mode="head"):
            self.df = df.reset_index(drop=True)
            self.tok = tok; self.max_len = max_len; self.mode = mode
        def __len__(self): return len(self.df)
        def __getitem__(self, idx):
            review = str(self.df.iloc[idx]["review"])
            if self.mode == "tail":
                full = self.tok(review, add_special_tokens=False)["input_ids"]
                keep = full[-(self.max_len - 2):]
                ids = [self.tok.cls_token_id] + keep + [self.tok.sep_token_id]
                attn = [1] * len(ids)
            else:
                enc = self.tok(review, max_length=self.max_len, truncation=True,
                               padding=False)
                ids, attn = enc["input_ids"], enc["attention_mask"]
            return {"input_ids": torch.tensor(ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attn, dtype=torch.long)}
    
    def collate(batch):
        max_l = max(b["input_ids"].size(0) for b in batch)
        pad_id = tokenizer.pad_token_id
        ids, attn = [], []
        for b in batch:
            L = b["input_ids"].size(0); pad = max_l - L
            ids.append(F.pad(b["input_ids"], (0, pad), value=pad_id))
            attn.append(F.pad(b["attention_mask"], (0, pad), value=0))
        return {"input_ids": torch.stack(ids), "attention_mask": torch.stack(attn)}
    
    runs = []
    model.eval()
    for mode in ("head", "tail"):
        ds = ScoreDS(df, tokenizer, max_len, mode=mode)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            collate_fn=collate, num_workers=0)
        probs = []
        with torch.no_grad():
            for batch in loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.cuda.amp.autocast():
                    out = model(**batch)
                p = F.softmax(out.logits, dim=-1)[:, 1].cpu().numpy()
                probs.extend(p)
        runs.append(np.array(probs))
    return np.mean(runs, axis=0)


def train_teacher_on_split(tr_df, va_df, ckpt_path):
    print(f"=== Training teacher split -> {ckpt_path} ===")

    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME, use_fast=True)

    class TeacherDS(Dataset):
        def __init__(self, df, tokenizer, max_len):
            self.df = df.reset_index(drop=True)
            self.tokenizer = tokenizer
            self.max_len = max_len

        def __len__(self):
            return len(self.df)

        def __getitem__(self, idx):
            row = self.df.iloc[idx]
            enc = self.tokenizer(
                str(row["review"]),
                max_length=self.max_len,
                truncation=True,
                padding=False,
            )
            return {
                "input_ids": torch.tensor(enc["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(enc["attention_mask"], dtype=torch.long),
                "labels": torch.tensor(label_to_num[row["label"]], dtype=torch.long),
            }

    def teacher_collate(batch):
        max_l = max(b["input_ids"].size(0) for b in batch)
        pad_id = teacher_tokenizer.pad_token_id

        ids, attn, labels = [], [], []
        for b in batch:
            L = b["input_ids"].size(0)
            pad = max_l - L
            ids.append(F.pad(b["input_ids"], (0, pad), value=pad_id))
            attn.append(F.pad(b["attention_mask"], (0, pad), value=0))
            labels.append(b["labels"])

        return {
            "input_ids": torch.stack(ids),
            "attention_mask": torch.stack(attn),
            "labels": torch.stack(labels),
        }

    train_ds = TeacherDS(tr_df, teacher_tokenizer, TEACHER_MAX_LEN)
    val_ds = TeacherDS(va_df, teacher_tokenizer, TEACHER_MAX_LEN)

    train_loader = DataLoader(
        train_ds,
        batch_size=TEACHER_BATCH,
        shuffle=True,
        collate_fn=teacher_collate,
        num_workers=0,
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=TEACHER_BATCH * 2,
        shuffle=False,
        collate_fn=teacher_collate,
        num_workers=0,
    )

    model = AutoModelForSequenceClassification.from_pretrained(
        TEACHER_NAME,
        num_labels=2,
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=TEACHER_LR, weight_decay=0.01)

    total_steps = len(train_loader) * TEACHER_EPOCHS
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.06 * total_steps),
        num_training_steps=total_steps,
    )

    use_amp = device.type == "cuda"
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    best_val = 0.0

    for epoch in range(TEACHER_EPOCHS):
        model.train()
        running, correct, total = 0.0, 0, 0

        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=use_amp):
                out = model(**batch)
                loss = out.loss

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            bs = batch["labels"].size(0)
            running += loss.item() * bs
            correct += (out.logits.argmax(-1) == batch["labels"]).sum().item()
            total += bs

        train_acc = correct / total

        model.eval()
        v_correct, v_total = 0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.cuda.amp.autocast(enabled=use_amp):
                    out = model(**batch)
                v_correct += (out.logits.argmax(-1) == batch["labels"]).sum().item()
                v_total += batch["labels"].size(0)

        val_acc = v_correct / v_total
        print(f"Teacher epoch {epoch+1}: train_acc={train_acc:.4f} val_acc={val_acc:.4f}")

        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), ckpt_path)
            print(f"  saved teacher checkpoint, val_acc={val_acc:.4f}")

    del model
    torch.cuda.empty_cache()

    return teacher_tokenizer


def generate_teacher_probs():
    print("=== Generating OOF teacher soft labels ===")

    teacher_tokenizer = AutoTokenizer.from_pretrained(TEACHER_NAME, use_fast=True)

    y = train_df["label"].map(label_to_num).values
    train_pos = np.zeros(len(train_df), dtype=np.float32)
    unsup_runs = []

    if USE_OOF_TEACHER and TRAIN_TEACHER:
        skf = StratifiedKFold(
            n_splits=TEACHER_N_FOLDS,
            shuffle=True,
            random_state=SEED,
        )

        for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, y)):
            print(f"\n=== Teacher fold {fold+1}/{TEACHER_N_FOLDS} ===")

            fold_ckpt = ART_DIR / f"teacher_{TEACHER_MODE}_fold{fold}.pt"

            tr_df = train_df.iloc[tr_idx].reset_index(drop=True)
            va_df = train_df.iloc[va_idx].reset_index(drop=True)

            if not fold_ckpt.exists():
                train_teacher_on_split(tr_df, va_df, fold_ckpt)
            else:
                print(f"Reusing teacher fold checkpoint: {fold_ckpt}")

            teacher = AutoModelForSequenceClassification.from_pretrained(
                TEACHER_NAME,
                num_labels=2,
            ).to(device)
            teacher.load_state_dict(torch.load(fold_ckpt, map_location=device))
            teacher.eval()

            print("Scoring held-out fold...")
            train_pos[va_idx] = teacher_score_corpus(
                va_df,
                teacher_tokenizer,
                teacher,
                max_len=TEACHER_MAX_LEN,
                batch_size=TEACHER_SCORE_BATCH,
            )

            if HAS_UNSUP:
                print(f"Scoring unsup with fold {fold+1} teacher...")
                unsup_fold = teacher_score_corpus(
                    unsup_df,
                    teacher_tokenizer,
                    teacher,
                    max_len=TEACHER_MAX_LEN,
                    batch_size=TEACHER_SCORE_BATCH,
                )
                unsup_runs.append(unsup_fold)

            del teacher
            torch.cuda.empty_cache()

        unsup_pos = np.mean(unsup_runs, axis=0) if HAS_UNSUP else np.array([])

    else:
        # Fallback: original single-teacher behavior
        if TRAIN_TEACHER:
            if not TEACHER_CKPT.exists():
                tr, va = train_test_split(
                    train_df,
                    test_size=0.05,
                    random_state=SEED,
                    stratify=train_df["label"],
                )
                train_teacher_on_split(tr, va, TEACHER_CKPT)

            teacher = AutoModelForSequenceClassification.from_pretrained(
                TEACHER_NAME,
                num_labels=2,
            ).to(device)
            teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=device))
        else:
            teacher = AutoModelForSequenceClassification.from_pretrained(
                TEACHER_NAME
            ).to(device)

        train_pos = teacher_score_corpus(
            train_df,
            teacher_tokenizer,
            teacher,
            max_len=TEACHER_MAX_LEN,
            batch_size=TEACHER_SCORE_BATCH,
        )

        if HAS_UNSUP:
            unsup_pos = teacher_score_corpus(
                unsup_df,
                teacher_tokenizer,
                teacher,
                max_len=TEACHER_MAX_LEN,
                batch_size=TEACHER_SCORE_BATCH,
            )
        else:
            unsup_pos = np.array([])

        del teacher
        torch.cuda.empty_cache()

    np.savez(TEACHER_PROBS_PATH, train_pos=train_pos, unsup_pos=unsup_pos)
    print(f"Saved teacher probs to {TEACHER_PROBS_PATH}")

    train_pred = (train_pos >= 0.5).astype(int)
    train_true = train_df["label"].map(label_to_num).values
    print(f"OOF teacher accuracy on train.csv: {(train_pred == train_true).mean():.4f}")

## 8. Distillation training utilities — KD loss, R-Drop, FGM, EMA

Each component is a small, composable helper. The training loop below stitches them together.

**KD loss.** Standard Hinton distillation. For labeled rows we mix in cross-entropy with `alpha`; for unsup rows we use pure soft loss.

**R-Drop.** Two forward passes per step with different dropout masks. Add symmetric KL between the two output distributions to the loss.

**FGM.** After the main backward pass, perturb the embedding-layer weights along the gradient direction with a small ε, recompute the loss, accumulate gradients, then restore the embeddings before the optimizer step.

**EMA.** Maintain a shadow copy of model weights with exponential decay. At inference, swap in the EMA weights.

In [ ]:
# ---------- KD loss ----------
def kd_loss(student_logits, teacher_pos_prob, hard_labels, T=3.0, alpha=0.3):
    """
    Combined distillation loss.
    
    student_logits:    (B, 2)
    teacher_pos_prob:  (B,) in [0, 1], or NaN for rows without teacher signal
    hard_labels:       (B,) in {0, 1}, or -1 for rows without hard labels
    """
    has_teacher = ~torch.isnan(teacher_pos_prob)
    has_hard = hard_labels != -1
    
    # Build teacher 2-class distribution: (1-p, p)
    teacher_probs_2 = torch.stack([1.0 - teacher_pos_prob, teacher_pos_prob], dim=-1)
    
    soft_loss = torch.tensor(0.0, device=student_logits.device)
    if has_teacher.any():
        s_logits = student_logits[has_teacher]
        t_probs  = teacher_probs_2[has_teacher].clamp(min=1e-7, max=1-1e-7)
        # Temperature-scaled KL
        s_log_soft = F.log_softmax(s_logits / T, dim=-1)
        # We treat teacher as a fixed distribution (already softmax'd at T=1);
        # to be temperature-consistent we re-temperature it here.
        t_logits   = torch.log(t_probs)
        t_soft     = F.softmax(t_logits / T, dim=-1)
        soft_loss = F.kl_div(s_log_soft, t_soft, reduction="batchmean") * (T * T)
    
    hard_loss = torch.tensor(0.0, device=student_logits.device)
    if has_hard.any():
        hard_loss = F.cross_entropy(student_logits[has_hard], hard_labels[has_hard])
    
    if has_teacher.any() and has_hard.any():
        return alpha * hard_loss + (1 - alpha) * soft_loss
    elif has_teacher.any():
        return soft_loss
    else:
        return hard_loss


# ---------- FGM ----------
class FGM:
    """Adversarial perturbation on the embedding layer."""
    def __init__(self, model, eps=1.0, emb_name="word_embeddings"):
        self.model = model
        self.eps = eps
        self.emb_name = emb_name
        self.backup = {}
    def attack(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad and self.emb_name in name and p.grad is not None:
                self.backup[name] = p.data.clone()
                norm = p.grad.norm()
                if norm != 0 and not torch.isnan(norm):
                    r = self.eps * p.grad / norm
                    p.data.add_(r)
    def restore(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad and self.emb_name in name and name in self.backup:
                p.data = self.backup[name]
        self.backup = {}


# ---------- EMA ----------
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone()
                       for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    def apply_to(self, model):
        """Swap model weights for EMA copy. Call restore() to undo."""
        self.backup = {}
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.backup[n] = p.data.clone()
                p.data.copy_(self.shadow[n])
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.backup:
                p.data.copy_(self.backup[n])
        self.backup = {}


# ---------- R-Drop term ----------
def rdrop_kl(logits1, logits2):
    p1 = F.log_softmax(logits1, dim=-1)
    p2 = F.log_softmax(logits2, dim=-1)
    return (F.kl_div(p1, p2.exp(), reduction="batchmean") +
            F.kl_div(p2, p1.exp(), reduction="batchmean")) / 2

## 9. Layer-wise LR decay (unchanged)

In [ ]:
def get_grouped_parameters(model, base_lr, layer_decay=0.85, weight_decay=0.01):
    no_decay = ("bias", "LayerNorm.weight", "layer_norm.weight")
    n_layers = model.backbone.config.num_hidden_layers

    def get_depth(name):
        if name.startswith("backbone.embeddings"):
            return 0
        if name.startswith("backbone.encoder.layer."):
            return int(name.split("backbone.encoder.layer.")[1].split(".")[0]) + 1
        return n_layers + 1

    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        depth = get_depth(name)
        lr = base_lr * (layer_decay ** (n_layers + 1 - depth))
        wd = 0.0 if any(nd in name for nd in no_decay) else weight_decay
        key = (lr, wd)
        groups.setdefault(key, []).append(param)
    return [{"params": p, "lr": lr, "weight_decay": wd} for (lr, wd), p in groups.items()]

## 10. The unified training loop

Combines KD + R-Drop + FGM + EMA. Each technique is gated by its master switch so you can ablate.

The loss per step:
1. Two student forward passes (different dropout) → KD losses + R-Drop KL term.
2. Backward.
3. FGM perturbs embeddings, second forward+backward, gradients accumulate.
4. Restore embeddings. Optimizer step. EMA update.

In [ ]:
def train_one_seed(
    train_df_in,
    val_df_in,
    seed,
    epochs=15,
    batch_size=64,
    base_lr=3e-5,
    warmup_ratio=0.1,
    layer_decay=0.85,
    patience=4,
    kd_T=3.0, kd_alpha=0.3,
    rdrop_alpha=1.0,
    fgm_eps=1.0,
    ema_decay=0.999,
    save_path="student.pt",
    verbose=True,
):
    set_seed(seed)
    
    train_ds = IMDBDataset(train_df_in, student_tokenizer)
    val_ds   = IMDBDataset(val_df_in,   student_tokenizer)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              collate_fn=student_collate, num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2, shuffle=False,
                              collate_fn=student_collate, num_workers=0)
    
    # Build student, optionally with TAPT-pretrained encoder
    model = MeanPoolBertClassifier(STUDENT_NAME, dropout=0.3).to(device)
    if USE_TAPT and TAPT_OUT.exists():
        model.load_encoder_state(TAPT_OUT)
    
    # Optimizer with layer-wise LR decay
    param_groups = get_grouped_parameters(model, base_lr=base_lr,
                                          layer_decay=layer_decay, weight_decay=0.01)
    optimizer = torch.optim.AdamW(param_groups)
    total_steps = len(train_loader) * epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(warmup_ratio * total_steps),
        num_training_steps=total_steps,
    )
    
    fgm = FGM(model, eps=fgm_eps) if USE_FGM else None
    ema = EMA(model, decay=ema_decay) if USE_EMA else None
    
    best_val = 0.0
    bad_epochs = 0
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    
    for epoch in range(epochs):
        # ----- TRAIN -----
        model.train()
        running, n = 0.0, 0
        for batch in train_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            optimizer.zero_grad()
            
            if USE_RDROP:
                out1 = model(input_ids=batch["input_ids"],
                             attention_mask=batch["attention_mask"])
                out2 = model(input_ids=batch["input_ids"],
                             attention_mask=batch["attention_mask"])
                logits1, logits2 = out1["logits"], out2["logits"]
                
                if USE_DISTILLATION:
                    l1 = kd_loss(logits1, batch["teacher_pos_prob"], batch["labels"],
                                 T=kd_T, alpha=kd_alpha)
                    l2 = kd_loss(logits2, batch["teacher_pos_prob"], batch["labels"],
                                 T=kd_T, alpha=kd_alpha)
                else:
                    has_hard = batch["labels"] != -1
                    l1 = F.cross_entropy(logits1[has_hard], batch["labels"][has_hard])
                    l2 = F.cross_entropy(logits2[has_hard], batch["labels"][has_hard])
                
                base = (l1 + l2) / 2
                kl_term = rdrop_kl(logits1, logits2)
                loss = base + rdrop_alpha * kl_term
            else:
                out = model(input_ids=batch["input_ids"],
                            attention_mask=batch["attention_mask"])
                logits = out["logits"]
                if USE_DISTILLATION:
                    loss = kd_loss(logits, batch["teacher_pos_prob"], batch["labels"],
                                   T=kd_T, alpha=kd_alpha)
                else:
                    has_hard = batch["labels"] != -1
                    loss = F.cross_entropy(logits[has_hard], batch["labels"][has_hard])
            
            loss.backward()
            
            # FGM adversarial second pass
            if USE_FGM:
                fgm.attack()
                out_adv = model(input_ids=batch["input_ids"],
                                attention_mask=batch["attention_mask"])
                if USE_DISTILLATION:
                    loss_adv = kd_loss(out_adv["logits"], batch["teacher_pos_prob"],
                                       batch["labels"], T=kd_T, alpha=kd_alpha)
                else:
                    has_hard = batch["labels"] != -1
                    loss_adv = F.cross_entropy(out_adv["logits"][has_hard],
                                               batch["labels"][has_hard])
                loss_adv.backward()
                fgm.restore()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            if USE_EMA:
                ema.update(model)
            
            bs = batch["input_ids"].size(0)
            running += loss.item() * bs
            n += bs
        
        train_loss = running / n
        
        # ----- VALIDATE (with EMA weights if enabled) -----
        if USE_EMA:
            ema.apply_to(model)
        
        model.eval()
        v_loss, v_correct, v_total = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                out = model(input_ids=batch["input_ids"],
                            attention_mask=batch["attention_mask"])
                has_hard = batch["labels"] != -1
                if has_hard.any():
                    l = F.cross_entropy(out["logits"][has_hard], batch["labels"][has_hard])
                    v_loss += l.item() * has_hard.sum().item()
                    pred = out["logits"][has_hard].argmax(-1)
                    v_correct += (pred == batch["labels"][has_hard]).sum().item()
                    v_total += has_hard.sum().item()
        val_loss = v_loss / max(v_total, 1)
        val_acc  = v_correct / max(v_total, 1)
        
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        
        if verbose:
            print(f"[seed={seed}] epoch {epoch+1}/{epochs} | "
                  f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")
        
        if val_acc > best_val:
            best_val = val_acc
            # Save EMA weights (already swapped in)
            torch.save(model.state_dict(), save_path)
            bad_epochs = 0
        else:
            bad_epochs += 1
        
        if USE_EMA:
            ema.restore(model)   # back to live weights for training
        
        if bad_epochs >= patience:
            if verbose:
                print(f"  early stopping at epoch {epoch+1}")
            break
    
    # Reload best (EMA) weights
    model.load_state_dict(torch.load(save_path, map_location=device))
    return model, best_val, history

## 11. Train the ensemble

Build the combined training corpus: labeled `train.csv` + balanced confident unsup pseudo-labels (with teacher soft probs). Train `N_ENSEMBLE_SEEDS` independent students.

In [ ]:
# Assemble the master training dataframe
train_df["label_int"] = train_df["label"].map(label_to_num)

if USE_DISTILLATION and len(unsup_kept) > 0:
    # Unsup rows: teacher_pos_prob present, label_int = -1 (no hard label)
    unsup_kept = unsup_kept.copy()
    unsup_kept["label_int"] = -1
    full_train_df = pd.concat([
        train_df[["review", "label_int", "teacher_pos_prob"]],
        unsup_kept[["review", "label_int", "teacher_pos_prob"]],
    ], ignore_index=True)
else:
    if "teacher_pos_prob" not in train_df.columns:
        train_df["teacher_pos_prob"] = float("nan")
    full_train_df = train_df[["review", "label_int", "teacher_pos_prob"]].copy()

print(f"Total training rows: {len(full_train_df)}")
print(f"  hard-labeled: {(full_train_df['label_int'] != -1).sum()}")
print(f"  pseudo-only:  {(full_train_df['label_int'] == -1).sum()}")

In [ ]:
SEEDS = [42, 2025, 7, 13, 1337, 0, 2026, 314, 2718, 9001, 123, 88, 999, 17, 5150][:N_ENSEMBLE_SEEDS]

ensemble_paths = []
val_records = []
for s in SEEDS:
    save_path = ART_DIR / f"student_seed{s}.pt"
    print(f"\n=== Training student, seed={s} ===")
    
    # 5% holdout from labeled-only rows for early stopping
    labeled_only = full_train_df[full_train_df["label_int"] != -1]
    pseudo_only  = full_train_df[full_train_df["label_int"] == -1]
    tr_lab, va_lab = train_test_split(labeled_only, test_size=0.05,
                                       random_state=s, stratify=labeled_only["label_int"])
    tr = pd.concat([tr_lab, pseudo_only], ignore_index=True)
    va = va_lab.reset_index(drop=True)
    val_records.append((str(save_path), va.copy()))
    
    model, va_acc, hist = train_one_seed(
        tr, va,
        seed=s,
        epochs=25,
        batch_size=64,
        base_lr=3e-5,
        warmup_ratio=0.1,
        layer_decay=0.85,
        patience=6,
        kd_T=3.0, kd_alpha=0.2,
        rdrop_alpha=1.0,
        fgm_eps=1.0,
        ema_decay=0.999,
        save_path=str(save_path),
    )
    print(f"Seed {s}: best val acc = {va_acc:.4f}")
    ensemble_paths.append(save_path)
    
    del model
    torch.cuda.empty_cache()

print(f"\nTrained {len(ensemble_paths)} ensemble members.")

## 12. Inference — TTA + ensemble averaging

Three crops per model (`head`, `middle`, `tail`) × `N_ENSEMBLE_SEEDS` models = `3N` averaged probabilities per test review.

In [ ]:
class SlidingWindowDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=MAX_LEN, stride=256):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.stride = stride
        self.items = []

        inner = max_len - 2

        for row_idx, review in enumerate(self.df["review"].astype(str).tolist()):
            full = tokenizer(review, add_special_tokens=False)["input_ids"]

            if len(full) <= inner:
                chunks = [full]
            else:
                starts = list(range(0, len(full) - inner + 1, stride))
                last_start = len(full) - inner
                if starts[-1] != last_start:
                    starts.append(last_start)
                chunks = [full[s:s + inner] for s in starts]

            for chunk in chunks:
                ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
                attn = [1] * len(ids)
                self.items.append((row_idx, ids, attn))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        row_idx, ids, attn = self.items[idx]
        return {
            "row_idx": torch.tensor(row_idx, dtype=torch.long),
            "input_ids": torch.tensor(ids, dtype=torch.long),
            "attention_mask": torch.tensor(attn, dtype=torch.long),
        }


def sliding_collate(batch):
    max_l = max(item["input_ids"].size(0) for item in batch)
    pad_id = student_tokenizer.pad_token_id

    row_idx, input_ids, attn = [], [], []

    for item in batch:
        L = item["input_ids"].size(0)
        pad = max_l - L

        row_idx.append(item["row_idx"])
        input_ids.append(F.pad(item["input_ids"], (0, pad), value=pad_id))
        attn.append(F.pad(item["attention_mask"], (0, pad), value=0))

    return {
        "row_idx": torch.stack(row_idx),
        "input_ids": torch.stack(input_ids),
        "attention_mask": torch.stack(attn),
    }


@torch.no_grad()
def get_sliding_probs(paths, df, batch_size=128, stride=256):
    all_model_probs = []

    ds = SlidingWindowDataset(df, student_tokenizer, max_len=MAX_LEN, stride=stride)
    loader = DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=sliding_collate,
        num_workers=0,
    )

    for path in paths:
        print(f"Predicting with {path}")

        m = MeanPoolBertClassifier(STUDENT_NAME, dropout=0.3).to(device)
        m.load_state_dict(torch.load(path, map_location=device))
        m.eval()

        sums = np.zeros(len(df), dtype=np.float64)
        counts = np.zeros(len(df), dtype=np.float64)

        for batch in loader:
            row_idx = batch["row_idx"].numpy()

            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)

            out = m(input_ids=input_ids, attention_mask=attention_mask)
            probs = F.softmax(out["logits"], dim=-1)[:, 1].cpu().numpy()

            np.add.at(sums, row_idx, probs)
            np.add.at(counts, row_idx, 1)

        model_probs = sums / np.maximum(counts, 1)
        all_model_probs.append(model_probs)

        del m
        torch.cuda.empty_cache()

    return np.mean(all_model_probs, axis=0)

# Make sure test_df has the columns IMDBDataset expects
test_df_for_pred = test_df.copy()
test_df_for_pred["label_int"] = -1
test_df_for_pred["teacher_pos_prob"] = float("nan")

test_probs = get_sliding_probs(
    ensemble_paths,
    test_df_for_pred,
    batch_size=128,
    stride=256,
)

# Tune threshold using each seed's validation split
val_probs_all = []
val_y_all = []

for path, va_df in val_records:
    va_df_for_pred = va_df.copy()
    va_df_for_pred["teacher_pos_prob"] = float("nan")

    probs = get_sliding_probs(
        [path],
        va_df_for_pred,
        batch_size=128,
        stride=256,
    )

    val_probs_all.append(probs)
    val_y_all.append(va_df_for_pred["label_int"].values)

val_probs_all = np.concatenate(val_probs_all)
val_y_all = np.concatenate(val_y_all)

best_t = 0.5
best_acc = 0.0

for t in np.linspace(0.35, 0.65, 301):
    pred = (val_probs_all >= t).astype(int)
    acc = (pred == val_y_all).mean()

    if acc > best_acc:
        best_acc = acc
        best_t = t

print(f"Best threshold = {best_t:.4f}, validation accuracy = {best_acc:.4f}")

test_preds = (test_probs >= best_t).astype(int)

submission = pd.DataFrame({
    "Id": test_df["id"].values,
    "Label": [num_to_label[p] for p in test_preds],
})
submission.to_csv("prediction.csv", index=False)
print("Wrote prediction.csv")
print(submission.head())
print()
print("Predicted label distribution:")
print(submission["Label"].value_counts())

## 13. Notes & ablation guide

### What to do if you stall below 0.94

1. **Strengthen the teacher.** Switch `TEACHER_MODE` from `"B"` → `"A"` (RoBERTa-large). Each percentage point of teacher accuracy typically translates to ~0.3-0.5% in the student.
2. **More TAPT epochs.** Bump `TAPT_EPOCHS` from 3 to 5. Cheap to retry.
3. **Tune KD temperature.** Try `kd_T = 2.0` and `kd_T = 4.0`. The sweet spot is dataset-dependent.
4. **Tune `kd_alpha`.** Lower alpha (0.1) emphasizes the teacher more, higher (0.5) the hard labels. With a strong teacher, lower is usually better.
5. **More ensemble members.** Bump `N_ENSEMBLE_SEEDS` to 10. Diminishing but real.
6. **Tighten unsup confidence.** In the section that builds `unsup_kept`, change `UNSUP_KEEP_LO/HI` from `0.1/0.9` to `0.05/0.95` or `0.02/0.98` for cleaner pseudo-labels (at the cost of fewer of them).
7. **Snapshot ensembling within a single seed.** Switch the LR scheduler from linear to `CosineAnnealingWarmRestarts` and save a checkpoint at each restart — you get 3-4 "models" per training run.

### Sanity checks before submitting

- `len(submission) == 5000` and `submission["Id"]` matches `test_df["id"]` exactly.
- Labels are lowercase `positive` / `negative` (the project rubric clarification).
- Predicted distribution is roughly balanced (~2500/2500). Severe imbalance signals a bug.

### Runtime estimates (single T4 GPU)
- TAPT (3 epochs, ~145k texts): ~30-45 min
- Teacher RoBERTa-base fine-tune (3 epochs): ~30-45 min  (large: 2-3 hours)
- Teacher scoring train + unsup: ~10-15 min
- Per student seed (15 epochs of ~95k examples with R-Drop+FGM): ~30-40 min
- 7-seed ensemble: ~3.5-5 hours
- **Total for full pipeline: ~5-7 hours on a T4**